In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
print(sys.path)

['/home/thule/miniconda3/envs/open-clip-training/lib/python311.zip', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/lib-dynload', '', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/site-packages', '../']


In [2]:
import os
import pandas as pd

In [3]:
column_names = ["timestamp", "arbitration_id", "dlc", "data0", "data1", "data2", "data3",
                 "data4", "data5", "data6", "data7", "attack"]

In [3]:
def fill_flag(sample):
    if not isinstance(sample['attack'], str):
        col = 'data' + str(sample['dlc'])
        sample['attack'], sample[col] = sample[col], sample['attack']
    return sample

In [4]:
d_path = '../data/car-hacking-attack-and-defense-challenge/0_Preliminary/1_Submission/Pre_submit_D.csv'
d_df = pd.read_csv(d_path)
print("Length of driving file: ", len(d_df))
print(d_df.head())

Length of driving file:  2000733
      Timestamp Arbitration_ID  DLC                     Data   Class SubClass
0  1.599047e+09            130    8  00 80 B0 80 00 00 0A 41  Normal   Normal
1  1.599047e+09            140    8  C8 7F 00 72 20 00 0A 30  Normal   Normal
2  1.599047e+09            260    8  05 26 01 30 00 BB 57 2B  Normal   Normal
3  1.599047e+09            329    8  4A BF 7D 8C 31 2A 00 10  Normal   Normal
4  1.599047e+09            251    8  F6 03 13 18 00 F9 F7 7F  Normal   Normal


In [5]:
d_df['SubClass'].unique()

<StringArray>
['Normal', 'Flooding', 'Spoofing', 'Replay', 'Fuzzing']
Length: 5, dtype: str

In [6]:
s_path = '../data/car-hacking-attack-and-defense-challenge/0_Preliminary/1_Submission/Pre_submit_S.csv'
s_df = pd.read_csv(s_path)
print("Length of stationary file: ", len(s_df))
print(s_df.head())

Length of stationary file:  1751313
      Timestamp Arbitration_ID  DLC                     Data   Class SubClass
0  1.599045e+09            436    4              00 00 00 04  Normal   Normal
1  1.599045e+09            453    5           00 88 91 00 EA  Normal   Normal
2  1.599045e+09            470    8  15 41 01 04 54 50 40 B5  Normal   Normal
3  1.599045e+09            47F    8  00 83 FF FA 00 78 00 8B  Normal   Normal
4  1.599045e+09            260    8  05 2F 00 30 00 BC 51 0A  Normal   Normal


In [7]:
s_df['SubClass'].unique()

<StringArray>
['Normal', 'Flooding', 'Spoofing', 'Replay', 'Fuzzing']
Length: 5, dtype: str

In [8]:
all = pd.concat([d_df, s_df], axis=0)
print("Length of all file: ", len(all))

Length of all file:  3752046


In [9]:
normal = all[all['Class'] == 'Normal']
print("Length of normal file: ", len(normal))
normal_copy = all[all['SubClass'] == 'Normal']
print("Length of normal file: ", len(normal_copy))
flooding = all[all['SubClass'] == 'Flooding']
print("Length of flooding file: ", len(flooding))
spoofing = all[all['SubClass'] == 'Spoofing']
print("Length of spoofing file: ", len(spoofing))
replay = all[all['SubClass'] == 'Replay']
print("Length of replay file: ", len(replay))
fuzzy = all[all['SubClass'] == 'Fuzzing']
print("Length of fuzzy file: ", len(fuzzy))

Length of normal file:  3358210
Length of normal file:  3358210
Length of flooding file:  191679
Length of spoofing file:  42583
Length of replay file:  62881
Length of fuzzy file:  96693


In [10]:
all.head(5)

,Timestamp,Arbitration_ID,DLC,Data,Class,SubClass
0,1.599047e+09,130,8,00 80 B0 80 00 00 0A 41,Normal,Normal
1,1.599047e+09,140,8,C8 7F 00 72 20 00 0A 30,Normal,Normal
2,1.599047e+09,260,8,05 26 01 30 00 BB 57 2B,Normal,Normal
3,1.599047e+09,329,8,4A BF 7D 8C 31 2A 00 10,Normal,Normal
4,1.599047e+09,251,8,F6 03 13 18 00 F9 F7 7F,Normal,Normal


In [11]:
all[all['DLC'] < 8].head(5)

,Timestamp,Arbitration_ID,DLC,Data,Class,SubClass
5,1.599047e+09,2B0,6,16 00 00 07 33 0B,Normal,Normal
9,1.599047e+09,164,4,00 08 1B A9,Normal,Normal
12,1.599047e+09,366,7,26 DC 0A 26 23 00 01,Normal,Normal
19,1.599047e+09,4F1,4,20 00 60 71,Normal,Normal
29,1.599047e+09,2B0,6,17 00 00 07 54 A6,Normal,Normal


In [12]:
# convert id from hex to dec
all['arbitration_id'] = all['Arbitration_ID'].apply(lambda x: int(x, 16) if isinstance(x, str) else x)


In [14]:
print(all.head(1))

      Timestamp Arbitration_ID  DLC                     Data   Class SubClass  \
0  1.599047e+09            130    8  00 80 B0 80 00 00 0A 41  Normal   Normal   

   arbitration_id  
0             304  


In [ ]:
# remove white space in Data string
all['data_field'] = all['Data'].str.replace(r'\s+', '', regex=True)

In [16]:
all[all['DLC'] < 8]['data_field'].head(5)

5       16000007330B
9           00081BA9
12    26DC0A26230001
19          20006071
29      1700000754A6
Name: data_field, dtype: str